# Phase 2: Feature Engineering

Build all features from raw data, then produce the train/test split used by all downstream notebooks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

## 1. Load & Clean

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df = df.drop(columns='customerID')
print(df.shape)

## 2. Tenure Grouping

In [ ]:
data = df.copy(deep=True)

bins = [0, 6, 12, 18, 24, 36, 48, 60, 72, 1000]
bin_labels = ['0-6', '6-12', '12-18', '18-24', '24-36', '36-48', '48-60', '60-72', '72+']
data['tenure_group'] = pd.cut(data['tenure'], bins=bins, labels=bin_labels, right=False)

## 3. Service Features

In [ ]:
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in services:
    data[col] = data[col].replace({'Yes': 1, 'No': 0, 'No internet service': 0})

data['HasInternet'] = (data['InternetService'] != 'No').astype(int)
data['NumServices'] = data[services].sum(axis=1)

data['MultipleLines'] = data['MultipleLines'].map({'No': 0, 'Yes': 1, 'No phone service': 0})
data['HasPhoneService'] = (data['PhoneService'] == 'Yes').astype(int)

## 4. Contract Features

In [ ]:
data['ContractMonths'] = data['Contract'].map({'Month-to-month': 1, 'One year': 12, 'Two year': 24})
data['IsMonthToMonth'] = (data['Contract'] == 'Month-to-month').astype(int)

# ContractProgress: fraction of contract completed (NaN for month-to-month)
data['ContractProgress'] = np.where(
    data['Contract'] != 'Month-to-month',
    (data['tenure'] / data['ContractMonths']).clip(upper=1),
    np.nan
)

print('IsMonthToMonth churn rates:')
data['Churn_num'] = data['Churn'].map({'Yes': 1, 'No': 0})
print(data.groupby('IsMonthToMonth')['Churn_num'].mean())
data = data.drop(columns='Churn_num')

## 5. Interaction Features

These operationalise the risk patterns found in EDA.

In [ ]:
# Fiber + month-to-month = highest-risk combination
data['fiber_contract_risk'] = (
    (data['InternetService'] == 'Fiber optic').astype(int) * data['IsMonthToMonth']
)

# Price pressure relative to time invested
data['price_tenure_risk'] = data['MonthlyCharges'] / (data['tenure'] + 1)

# New customer flag (first 6 months = highest-churn window)
data['early_customer'] = (data['tenure'] <= 6).astype(int)

# Monthly charge amplified by contract risk
data['price_stress'] = data['MonthlyCharges'] * data['IsMonthToMonth']

# Fiber + new customer = very high risk
data['fiber_new_customer'] = (
    (data['InternetService'] == 'Fiber optic').astype(int) * data['early_customer']
)

## 6. Encoding

In [ ]:
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})
data['gender'] = data['gender'].map({'Male': 0, 'Female': 1})
data['SeniorCitizen'] = data['SeniorCitizen'].map({'No': 0, 'Yes': 1})

for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    data[col] = data[col].map({'No': 0, 'Yes': 1})

# One-hot encode multi-category columns
data = pd.get_dummies(data, columns=['InternetService', 'PaymentMethod', 'Contract'], drop_first=True)

# Encode tenure_group as ordered integer
data['tenure_group'] = data['tenure_group'].cat.as_ordered()
data['tenure_group'] = data['tenure_group'].cat.codes

# Convert bool columns to int
bool_cols = data.select_dtypes(include='bool').columns
data[bool_cols] = data[bool_cols].astype(int)

# Fill ContractProgress NaN (month-to-month customers get 0)
data['ContractProgress'] = data['ContractProgress'].fillna(0)

print(data.shape)
data.info()

## 7. Train / Test Split

In [ ]:
X = data.drop('Churn', axis=1)
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Churn rate train: {y_train.mean():.3f}, test: {y_test.mean():.3f}')

## 8. Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## 9. SMOTE (training set only)

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f'After SMOTE: {X_train_resampled.shape}, class balance: {pd.Series(y_train_resampled).value_counts().to_dict()}')

## 10. Save Artefacts

In [ ]:
data.to_csv('../data/churn_features.csv', index=False)
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X_train.columns.tolist(), '../models/feature_columns.pkl')

# Save splits for use by modeling notebook
import numpy as np
np.save('../data/X_train.npy', X_train_resampled)
np.save('../data/X_test.npy', X_test_scaled.values)
np.save('../data/y_train.npy', y_train_resampled)
np.save('../data/y_test.npy', y_test.values)

# Also save unscaled splits (for tree models that don't need scaling)
np.save('../data/X_train_raw.npy', X_train.values)
np.save('../data/X_test_raw.npy', X_test.values)
np.save('../data/y_train_raw.npy', y_train.values)

print('Saved: churn_features.csv, scaler.pkl, feature_columns.pkl, splits')